# b02에서 시간 연결이 원판 궤적을 복구했나

바와 몸이 접촉하기 직전, 한 장에서 놓친 원판을 앞선 움직임으로 연결할 수 있는지 본다. 97·98번은 잘못 잡은 박스가 다음 선택에 미치는 영향을 볼 보조 사례다.

**실험 조건** · 검증 영상 `b02_sn_h_02`, 원판 베이스라인 `best.pt`, Mac CPU, 입력 640. 두 방식이 같은 후보를 받는다. 신뢰도 0.10 이상은 직접 검출한다. 연결을 켜면 마지막 직접 검출 뒤 최대 2프레임까지, 예상 중심에서 직전 박스 너비의 0.5배 안에 있는 저신뢰 후보(하한 0.001) 중 가장 가까운 것을 채택한다. 신뢰도는 원래 값으로 남긴다.

위에서부터 실행한다. 영상·이미지·CSV는 로컬 `outputs/video_analysis/b02_link_comparison/`에 저장한다.

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Video, display

from wlcv.analyze import read_phases
from wlcv.annotation.storage import load_events
from wlcv.config import ANNOTATIONS, ROOT, RUNS, VIDEOS
from wlcv.evaluation.full_video import compare_videos, detect_videos, summarize_video_scope
from wlcv.evaluation.link_replay import audit_candidates, phase_table, save_comparison, trajectory

video = "b02_sn_h_02"
weights = RUNS / "plate_baseline/weights/best.pt"
focus = [35, 97, 98]
lift, knee = read_phases(video)
drive = load_events(ANNOTATIONS, video, pass_number=1)["drive"]
print(f"수동 이륙 {lift}번 · 무릎 통과 {knee}번 · 바 접촉 {drive}번")

## 1. 같은 후보로 두 방식을 실행하기

0번부터 끝까지 추적한 뒤에 관심 구간을 자른다. 두 방식 모두 후보를 고른 다음 수동 검수 박스와 IoU를 계산한다. IoU 0.50 이상을 정탐으로 센다. 수동 박스가 없는 장면은 판독 불가로 남긴다.

In [ ]:
frames, candidates = detect_videos(weights, [video])
review = pd.read_csv(ANNOTATIONS / "b02_c07_plate_review.csv")
review = review[review.video_id == video]
detail, checks = compare_videos(frames, candidates, review, gate_high_history=False)

output = ROOT / "outputs/video_analysis/b02_link_comparison" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
detail.to_csv(output / "per_frame.csv", index=False)
checks.to_csv(output / "distance_checks.csv", index=False)
print(f"원본 {len(frames)}프레임 · 결과 폴더 {output}")

## 2. 바 접촉 전 궤적과 상승량 비교하기

보라색 점선은 수동 박스 중심, 초록 선은 선택한 원판 중심이다. 연결한 후보에 마름모를 표시한다. 가운데가 비더라도 양 끝의 선택이 같으면 구간 상승량은 같을 수 있다. 상승량은 시작 y에서 끝 y를 뺀 원본 화면의 px다.

In [ ]:
phases = phase_table(detail, lift, knee, drive)
phases.to_csv(output / "phases.csv", index=False)
display(phases.round(2))
fig = trajectory(detail, knee, drive)
fig.savefig(output / "trajectory.png", dpi=150)
plt.show()

## 3. 35번 복구와 97·98번 실패를 같은 프레임에서 비교하기

표에서 `high`는 직접 검출, `linked`는 시간 연결, `missing`은 선택 없음이다. 진단 표의 `best_candidate_iou`는 수동 박스에 가장 잘 맞는 후보를 **사후에** 찾아 계산한 값이다. 첫 표는 채택한 박스를, 두 번째 표는 모든 후보를 평가한다. 첫 표의 y 차이는 선택한 중심 y에서 수동 박스 중심 y를 뺀 값이다. 양수이면 화면에서 더 아래다. 거리와 반경은 px 단위다.

In [ ]:
cases = detail[detail.frame_index.isin(focus)].copy()
cases["center_y_error_px"] = cases.cy - (cases.gt_y1 + cases.gt_y2) / 2
cases.to_csv(output / "focus_frames.csv", index=False)
display(cases[["frame_index", "method", "source", "confidence", "iou", "match", "center_y_error_px"]].round(3))
audit = audit_candidates(detail, candidates, checks, focus)
audit.to_csv(output / "case_audit.csv", index=False)
display(audit.round(3))

양쪽의 자홍색 테두리는 수동 박스다. 모델 박스는 직접 검출이면 초록색, 시간 연결이면 주황색이다. 오른쪽 하늘색 원은 저신뢰 후보를 찾은 반경이다. 35번에서는 두 칸의 차이를, 97번에서는 두 칸이 함께 고른 위치를, 98번에서는 후보가 남았는데도 연결되지 않은 이유를 본다.

In [ ]:
timing = pd.read_csv(ROOT / "data/metadata/video_timing.csv").set_index("video_id")
fps = float(timing.at[video, "fps_from_median_gap"])
movie = save_comparison(VIDEOS / f"{video}.mp4", detail, checks, fps, focus, output)
for index in focus:
    display(Image(filename=str(output / f"frame_{index:03}.png"), width=570))
display(Video(filename=str(movie), embed=True, width=640))

## 4. 주 평가와 보조 구간을 따로 읽기

0–53번은 기존 주 평가 구간이다. 54–115번은 보조 구간이며 79–91번은 수동 박스를 정할 수 없었다. 아래 표는 판독 가능한 프레임만 정탐과 연속 미탐으로 센다. `wrong_link`는 저신뢰 후보를 연결했으나 검수 IoU가 기준에 못 미친 수다. 97번처럼 직접 검출한 박스의 오탐은 프레임별 표에서 별도로 확인한다.

In [ ]:
summary = summarize_video_scope(detail)
summary.to_csv(output / "scope_summary.csv", index=False)
display(summary[["scope", "method", "frames", "scored", "hits", "longest_miss", "recovered", "wrong_link"]])
print(output)